In [ ]:
#
# Ładowanie csv do duckdb
#
import duckdb
import kagglehub
import glob
import os

race_data  = kagglehub.dataset_download("jtrotman/formula-1-race-data")
event_data = kagglehub.dataset_download("jtrotman/formula-1-race-events")

conn = duckdb.connect("f1.db")

# Załaduj każdy CSV jako osobną tabelę
for f in glob.glob(f"{race_data}/*.csv"):
    name = os.path.splitext(os.path.basename(f))[0]
    conn.execute(f"""
        CREATE OR REPLACE TABLE {name} 
        AS SELECT * FROM read_csv_auto('{f}', null_padding=True,
            nullstr='\\N',
            sample_size=-1)
    """)
    print(f"✓ {name}")

# Events osobno
for f in glob.glob(f"{event_data}/*.csv"):
    name = "ev_" + os.path.splitext(os.path.basename(f))[0]
    conn.execute(f"""
        CREATE OR REPLACE TABLE {name}
        AS SELECT * FROM read_csv_auto('{f}', null_padding=True,
            nullstr='\\N',
            sample_size=-1)
    """)

conn.execute("SHOW TABLES").df()
conn.close()